# Lesson 7A: Anomaly Detection - Theory

## Finding the Unusual

**Case Study**: PayPal processes billions of transactions yearly. How do they identify the 0.1% that are fraudulent?

**Anomaly detection** finds data points significantly different from the majority.

**Applications**:
- 💳 **Fraud Detection**: Credit cards, insurance claims
- 🔒 **Network Security**: Intrusion detection, DDoS attacks
- 🏥 **Medical Diagnosis**: Disease outbreak, patient monitoring
- 🏭 **Equipment Failure**: Predictive maintenance, quality control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.datasets import make_blobs

np.random.seed(42)
print("✅ Loaded!")

## Method 1: Isolation Forest

**Intuition**: Anomalies are few and different → easier to isolate with random splits.

**Algorithm**:
1. Randomly select feature and split value
2. Recursively partition data
3. **Anomaly score** = average path length (shorter = more anomalous)

**Key insight**: Normal points require many splits to isolate, anomalies require few.

In [ ]:
# Isolation Forest from scratch (simplified)
class SimpleIsolationTree:
    def __init__(self, max_depth=10):
        self.max_depth = max_depth
        self.split_feature = None
        self.split_value = None
    
    def fit(self, X, depth=0):
        n_samples, n_features = X.shape
        
        if depth >= self.max_depth or n_samples <= 1:
            return depth
        
        # Random split
        self.split_feature = np.random.randint(n_features)
        feature_values = X[:, self.split_feature]
        self.split_value = np.random.uniform(feature_values.min(), feature_values.max())
        
        # Partition and recurse
        left_mask = feature_values < self.split_value
        if left_mask.sum() == 0 or (~left_mask).sum() == 0:
            return depth
        
        left_depth = self.fit(X[left_mask], depth + 1)
        right_depth = self.fit(X[~left_mask], depth + 1)
        
        return (left_depth + right_depth) / 2

# Demonstrate on simple data
X_normal = np.random.randn(100, 2) * 0.5
X_anomaly = np.array([[4, 4], [-4, -4]])
X = np.vstack([X_normal, X_anomaly])

# Build ensemble of trees
n_trees = 10
avg_depths = []
for i in range(len(X)):
    depths = []
    for _ in range(n_trees):
        tree = SimpleIsolationTree(max_depth=8)
        depth = tree.fit(X, 0)
        depths.append(depth)
    avg_depths.append(np.mean(depths))

avg_depths = np.array(avg_depths)
anomaly_scores = 1 / (avg_depths + 1)  # Shorter path = higher score

print(f"Normal point avg depth: {avg_depths[0]:.2f}")
print(f"Anomaly point avg depth: {avg_depths[-1]:.2f}")
print(f"✅ Anomalies have shorter paths!")

## Method 2: Local Outlier Factor (LOF)

**Intuition**: Anomalies have lower density than their neighbors.

**Algorithm**:
1. For each point, find k-nearest neighbors
2. Compute **local reachability density** (LRD)
3. **LOF** = ratio of neighbor density to point density

**Key insight**: LOF > 1 means point is in lower density region than neighbors.

In [ ]:
# LOF concept visualization
from sklearn.neighbors import NearestNeighbors

# Create data with local anomaly
X_dense = np.random.randn(200, 2) * 0.3 + [0, 0]
X_sparse = np.random.randn(50, 2) * 0.3 + [3, 3]
X_outlier = np.array([[3, 0]])
X = np.vstack([X_dense, X_sparse, X_outlier])

# Compute k-nearest neighbors
k = 10
nbrs = NearestNeighbors(n_neighbors=k+1).fit(X)
distances, indices = nbrs.kneighbors(X)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_dense[:, 0], X_dense[:, 1], c='blue', s=30, alpha=0.6, label='Dense cluster')
plt.scatter(X_sparse[:, 0], X_sparse[:, 1], c='green', s=30, alpha=0.6, label='Sparse cluster')
plt.scatter(X_outlier[:, 0], X_outlier[:, 1], c='red', s=200, marker='*', 
           edgecolors='black', linewidths=2, label='Outlier')

# Show neighbors of outlier
outlier_idx = len(X) - 1
neighbor_indices = indices[outlier_idx, 1:]
plt.scatter(X[neighbor_indices, 0], X[neighbor_indices, 1], 
           facecolors='none', edgecolors='red', s=100, linewidths=2)

plt.title('LOF: Outlier has distant neighbors', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Compute LOF
lof = LocalOutlierFactor(n_neighbors=k)
lof_scores = lof.fit_predict(X)
negative_factors = lof.negative_outlier_factor_

print(f"Dense cluster point LOF: {-negative_factors[0]:.2f}")
print(f"Sparse cluster point LOF: {-negative_factors[200]:.2f}")
print(f"Outlier LOF: {-negative_factors[-1]:.2f}")
print("✅ Higher LOF = more anomalous!")

## Method 3: One-Class SVM

**Intuition**: Learn decision boundary that encompasses normal data.

**Algorithm**:
1. Map data to high-dimensional space (kernel trick)
2. Find hyperplane separating origin from data
3. **nu parameter**: Expected fraction of anomalies

**Key insight**: Anomalies lie outside the learned boundary.

In [ ]:
# Create data with anomalies
X, _ = make_blobs(n_samples=300, centers=1, cluster_std=0.5, random_state=42)
X_anomalies = np.random.uniform(-5, 5, (20, 2))
X = np.vstack([X, X_anomalies])

# Compare all three methods
models = {
    'Isolation Forest': IsolationForest(contamination=0.1, random_state=42),
    'LOF': LocalOutlierFactor(contamination=0.1),
    'One-Class SVM': OneClassSVM(nu=0.1, gamma='auto')
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, model) in zip(axes, models.items()):
    if name == 'LOF':
        labels = model.fit_predict(X)
    else:
        labels = model.fit(X).predict(X)
    
    # Plot
    normal = labels == 1
    anomaly = labels == -1
    
    ax.scatter(X[normal, 0], X[normal, 1], c='blue', s=30, alpha=0.6, label='Normal')
    ax.scatter(X[anomaly, 0], X[anomaly, 1], c='red', s=80, marker='x', 
              linewidths=2, label='Anomaly')
    ax.set_title(name, fontweight='bold', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Anomaly Detection Methods Comparison', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

print(f"✅ Detected anomalies:")
for name, model in models.items():
    if name == 'LOF':
        labels = model.fit_predict(X)
    else:
        labels = model.fit(X).predict(X)
    print(f"   {name}: {sum(labels == -1)} anomalies")

## When to Use Each Method?

| Method | Best For | Pros | Cons |
|--------|----------|------|------|
| **Isolation Forest** | High-dimensional data, global anomalies | Fast, scales well, no distance metric needed | May miss local anomalies |
| **LOF** | Local anomalies, varying densities | Finds density-based outliers | Slow on large datasets |
| **One-Class SVM** | Complex boundaries, kernel methods | Flexible decision boundary | Sensitive to parameters, slow |

**Production tip**: Start with Isolation Forest for speed, use LOF for high-precision scenarios.